# Dana Bakery: deterministic LangGraph orders
Run cells in order in Google Colab when ready. This notebook is saved unexecuted. No LLM calls or API keys are used.

Each order uses a fixed independent inventory snapshot. Alternatives are the next two products in the stated order, without wrapping. See prompt.md for the final specification.

In [ ]:
%pip install -q langgraph


## 1. Catalog and state
The Annotated reducer appends each node’s new steps in processing order. Decimal keeps money calculations exact.

In [ ]:
import operator
from decimal import Decimal, ROUND_HALF_UP
from typing import Annotated, Any, Literal, TypedDict
from langgraph.graph import StateGraph, START, END

CATALOG = {
    "challah": (Decimal("18"), 10),
    "babka": (Decimal("45"), 0),
    "rugelach": (Decimal("30"), 3),
    "sourdough": (Decimal("25"), 4),
}
ALTERNATIVES = ["babka", "rugelach", "sourdough", "challah"]
DAYS = {"monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"}

def money(value):
    return value.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

class OrderState(TypedDict, total=False):
    order: dict[str, Any]
    requested: str
    item: str
    quantity: int
    day: str
    city: str
    error: str
    candidates: list[str]
    attempt: int
    available: bool
    subtotal: Decimal
    discount: Decimal
    goods_total: Decimal
    delivery: Literal["pickup", "delivery"]
    converted: bool
    total: float
    status: str
    message: str
    steps: Annotated[list[str], operator.add]


## 2. Validation and stock
Quantity must be an actual integer, so True and False are rejected. One alternatives-node visit checks one candidate.

In [ ]:
def validate(state):
    order = state["order"]
    raw_item = order.get("item")
    item = raw_item.strip().lower() if isinstance(raw_item, str) else ""
    update = {"requested": item, "item": item, "error": "", "steps": ["validate"]}
    if not all(key in order for key in ("item", "quantity", "day", "city")):
        return {**update, "error": "Please provide item, quantity, day, and city."}
    if item in {"croissant", "croissants"}:
        return {**update, "error": "Sorry, croissants are discontinued."}
    if item not in CATALOG:
        return {**update, "error": "Sorry, that product is not on our menu."}
    quantity = order["quantity"]
    if type(quantity) is not int or quantity <= 0:
        return {**update, "error": "Please enter a positive whole-number quantity."}
    day = order["day"]
    if not isinstance(day, str) or day.strip().lower() not in DAYS:
        return {**update, "error": "Please enter a valid English weekday."}
    if not isinstance(order["city"], str):
        return {**update, "error": "Please enter a city or an empty string for pickup."}
    return {**update, "quantity": quantity, "day": day.strip().lower(),
            "city": order["city"].strip()}

def inventory(state):
    item = state["item"]
    available = CATALOG[item][1] >= state["quantity"]
    position = ALTERNATIVES.index(item)
    # Only the next two products are candidates; the list never wraps.
    return {"available": available, "attempt": 0,
            "candidates": ALTERNATIVES[position + 1:position + 3],
            "steps": [f"inventory:{item}:{available}"]}

def alternative(state):
    candidate = state["candidates"][state["attempt"]]
    available = CATALOG[candidate][1] >= state["quantity"]
    update = {"available": available, "attempt": state["attempt"] + 1,
              "steps": [f"alternative:{candidate}:{available}"]}
    if available:
        update["item"] = candidate
    return update

def reject(state):
    reason = state.get("error") or "Sorry, stock and the allowed alternatives cannot cover your full quantity."
    # Rejected orders use pickup as a schema default; no fulfillment is arranged.
    return {"status": "rejected", "delivery": "pickup", "total": 0.0,
            "message": "Your order was rejected. " + reason + " Please try another order; we would love to help!",
            "steps": ["reject"]}


## 3. Price, delivery, and customer message
Only baked goods receive a discount. Delivery is added afterward. Customer messages use fixed templates.
The delivery output is exactly "pickup" or "delivery". Detailed location and fee explanations stay in message. Rejected orders use "pickup" only as a valid schema default; status and message explicitly state rejection, and no fulfillment is arranged.

In [ ]:
def price(state):
    subtotal = CATALOG[state["item"]][0] * state["quantity"]
    eligible = subtotal > Decimal("150") and state["day"] != "friday"
    discount = money(subtotal * Decimal("0.15")) if eligible else Decimal("0.00")
    return {"subtotal": money(subtotal), "discount": discount,
            "goods_total": money(subtotal - discount),
            "steps": ["price:discount" if eligible else "price:no_discount"]}

def delivery(state):
    city = state["city"]
    jerusalem = city.casefold() == "jerusalem"
    converted = bool(city) and not jerusalem
    fee = Decimal("20.00") if jerusalem else Decimal("0.00")
    step = "jerusalem" if jerusalem else ("converted_pickup" if converted else "pickup")
    return {"delivery": "delivery" if jerusalem else "pickup",
            "converted": converted, "total": float(money(state["goods_total"] + fee)),
            "steps": [f"delivery:{step}"]}

def message(state):
    parts = [f"Thank you! Your order for {state['quantity']} {state['item']} is confirmed."]
    if state["item"] != state["requested"]:
        parts.append(f"We substituted {state['item']} for {state['requested']} because stock could not cover your full quantity.")
    if state["discount"] > 0:
        parts.append(f"You saved {state['discount']:.2f} NIS with our 15% baked-goods discount!")
    if state["converted"]:
        parts.append(f"We do not deliver to {state['city']}, so your order has been changed to free pickup.")
    elif state["delivery"] == "delivery":
        parts.append("Jerusalem delivery costs 20.00 NIS.")
    else:
        parts.append("Your order is arranged for free pickup.")
    parts.append(f"Your total is {state['total']:.2f} NIS. Enjoy!")
    return {"status": "accepted", "message": " ".join(parts), "steps": ["message"]}


## 4. Graph and public helper
Conditional edges reject invalid orders, choose alternatives, and stop the search after at most two attempts. Nodes return only new steps; the reducer preserves earlier ones.

In [ ]:
def after_validate(state):
    return "reject" if state["error"] else "inventory"

def after_inventory(state):
    if state["available"]:
        return "price"
    return "alternative" if state["candidates"] else "reject"

def after_alternative(state):
    if state["available"]:
        return "price"
    return "alternative" if state["attempt"] < len(state["candidates"]) else "reject"

builder = StateGraph(OrderState)
for name, function in [("validate", validate), ("inventory", inventory),
                       ("alternative", alternative), ("reject", reject),
                       ("price", price), ("delivery", delivery), ("message", message)]:
    builder.add_node(name, function)
builder.add_edge(START, "validate")
builder.add_conditional_edges("validate", after_validate,
                              {"reject": "reject", "inventory": "inventory"})
routes = {"price": "price", "alternative": "alternative", "reject": "reject"}
builder.add_conditional_edges("inventory", after_inventory, routes)
builder.add_conditional_edges("alternative", after_alternative, routes)
builder.add_edge("price", "delivery")
builder.add_edge("delivery", "message")
builder.add_edge("message", END)
builder.add_edge("reject", END)
graph = builder.compile()

RESULT_FIELDS = ("status", "item", "delivery", "total", "message", "steps")
def process_order(order: dict[str, Any]) -> dict:
    """Process one independent order and return all six public fields."""
    state = graph.invoke({"order": dict(order), "steps": []})
    return {key: state[key] for key in RESULT_FIELDS}


## 5. Graph visualization
The default Mermaid PNG renderer may require internet access, without API keys. This optional visualization does not affect local order processing.

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))


## 6. Test orders
These tests are provided for later execution. Expected totals are calculated independently. The exact 150 NIS boundary is tested directly in the pricing node because available stock cannot produce that subtotal.

In [ ]:
def order(item="challah", quantity=1, day="Monday", city=""):
    return dict(item=item, quantity=quantity, day=day, city=city)

# name, input, expected status, item, delivery, total, message fragment
cases = [
    ("direct delivery", order(city="Jerusalem"), "accepted", "challah", "delivery", 38.0, "delivery costs"),
    ("first alternative", order("babka", 2), "accepted", "rugelach", "pickup", 60.0, "substituted"),
    ("second alternative", order("babka", 4), "accepted", "sourdough", "pickup", 100.0, "substituted"),
    ("two attempts exhausted", order("babka", 5), "rejected", "babka", "pickup", 0.0, "full quantity"),
    ("last alternative succeeds", order("sourdough", 5), "accepted", "challah", "pickup", 90.0, "substituted"),
    ("last alternative fails", order("sourdough", 11), "rejected", "sourdough", "pickup", 0.0, "full quantity"),
    ("no alternatives", order("challah", 11), "rejected", "challah", "pickup", 0.0, "full quantity"),
    ("discount plus delivery", order(quantity=10, city="Jerusalem"), "accepted", "challah", "delivery", 173.0, "saved 27.00"),
    ("Friday no discount", order(quantity=10, day="Friday", city="Jerusalem"), "accepted", "challah", "delivery", 200.0, "200.00"),
    ("converted pickup", order(city="Tel Aviv"), "accepted", "challah", "pickup", 18.0, "changed to free pickup"),
    ("normalized fields", order(" CHALLAH ", 1, " FRIDAY ", " jerusalem "), "accepted", "challah", "delivery", 38.0, "confirmed"),
    ("rejected Jerusalem order", order("challah", 11, city="Jerusalem"), "rejected", "challah", "pickup", 0.0, "full quantity"),
    ("discontinued", order("croissants"), "rejected", "croissants", "pickup", 0.0, "discontinued"),
    ("unknown item", order("cake"), "rejected", "cake", "pickup", 0.0, "not on our menu"),
    ("invalid weekday", order(day="Funday"), "rejected", "challah", "pickup", 0.0, "valid English weekday"),
    ("invalid city", order(city=None), "rejected", "challah", "pickup", 0.0, "enter a city"),
    ("missing field", {"item": "challah", "quantity": 1, "day": "Monday"}, "rejected", "challah", "pickup", 0.0, "provide item"),
]
for quantity in (0, -1, 1.5, "2", True, None):
    cases.append((f"invalid quantity {quantity!r}", order(quantity=quantity),
                  "rejected", "challah", "pickup", 0.0, "positive whole-number"))

for name, request, status, item, fulfillment, total, fragment in cases:
    result = process_order(request)
    assert set(result) == set(RESULT_FIELDS), name
    assert (result["status"], result["item"], result["delivery"], result["total"]) == (status, item, fulfillment, total), name
    assert result["delivery"] in {"pickup", "delivery"}, name
    assert fragment in result["message"], name
    if status == "rejected":
        assert result["delivery"] == "pickup", name
        assert "Your order was rejected." in result["message"], name
    elif fulfillment == "delivery":
        assert "Jerusalem delivery costs 20.00 NIS." in result["message"], name
    else:
        assert "free pickup" in result["message"], name
    assert result["steps"][0] == "validate", name
    assert result["steps"][-1] == ("message" if status == "accepted" else "reject"), name
    assert sum(step.startswith("alternative:") for step in result["steps"]) <= 2, name
    assert process_order(request) == result, name  # Identical input, identical output.
    print(name, result)

# Exact route assertions verify reducer order and the bounded loop.
assert process_order(order("babka", 4))["steps"] == [
    "validate", "inventory:babka:False", "alternative:rugelach:False",
    "alternative:sourdough:True", "price:no_discount", "delivery:pickup", "message",
]
assert process_order(order("babka", 5))["steps"] == [
    "validate", "inventory:babka:False", "alternative:rugelach:False",
    "alternative:sourdough:False", "reject",
]
# Unit checks bypass inventory solely to isolate the strict pricing boundary.
assert price({"item": "rugelach", "quantity": 5, "day": "monday"})["discount"] == Decimal("0.00")
assert price({"item": "rugelach", "quantity": 6, "day": "monday"})["goods_total"] == Decimal("153.00")
assert price({"item": "rugelach", "quantity": 6, "day": "friday"})["discount"] == Decimal("0.00")
assert "saved" not in process_order(order(quantity=10, day="Friday"))["message"]
print(f"All {len(cases)} order cases and extra routing/pricing checks passed.")
